In [1]:
!pip install sdv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
import warnings
warnings.filterwarnings("ignore")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/ransomware_project'
TARGET   = "label_encoded"

df_selected    = pd.read_csv(f'{save_dir}/df_selected.csv')
df_transformed = pd.read_csv(f'{save_dir}/df_transformed.csv')

print("df_selected   :", df_selected.shape)
print("df_transformed:", df_transformed.shape)

In [ ]:
def augmenter_ctgan(df, target_col, n_synthetic=500, epochs=300):
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(df)
    metadata.update_column(column_name=target_col, sdtype='categorical')

    ctgan = CTGANSynthesizer(metadata, epochs=epochs, batch_size=500, cuda=True)
    ctgan.fit(df)
    synthetic = ctgan.sample(num_rows=n_synthetic)
    df_aug = pd.concat([df, synthetic], ignore_index=True)
    print(f"  {len(df)} -> {len(df_aug)} lignes (+{n_synthetic} synthetiques)")
    return df_aug

print("Augmentation df_selected...")
df_selected_ctgan = augmenter_ctgan(df_selected, TARGET)

print("Augmentation df_transformed...")
df_transformed_ctgan = augmenter_ctgan(df_transformed, TARGET)

In [ ]:
df_selected_ctgan.to_csv(f'{save_dir}/df_selected_ctgan.csv', index=False)
df_transformed_ctgan.to_csv(f'{save_dir}/df_transformed_ctgan.csv', index=False)
print("Datasets sauvegardes sur Drive")

In [ ]:
def preparer(df):
    X = StandardScaler().fit_transform(df.drop(columns=[TARGET]).values)
    y = LabelEncoder().fit_transform(df[TARGET].values)
    return X, y

X_sel,   y_sel   = preparer(df_selected)
X_sel_c, y_sel_c = preparer(df_selected_ctgan)
X_tra,   y_tra   = preparer(df_transformed)
X_tra_c, y_tra_c = preparer(df_transformed_ctgan)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultats_ctgan = {}

def evaluer(nom, model, X_sel, y_sel, X_sel_aug, y_sel_aug,
                        X_tra, y_tra, X_tra_aug, y_tra_aug):
    def scores(X, y):
        preds = cross_val_predict(model, X, y, cv=cv)
        return {
            "Acc": round(accuracy_score(y, preds), 4),
            "F1" : round(f1_score(y, preds, average='weighted'), 4),
        }
    s1 = scores(X_sel,     y_sel)
    s2 = scores(X_sel_aug, y_sel_aug)
    s3 = scores(X_tra,     y_tra)
    s4 = scores(X_tra_aug, y_tra_aug)

    print(f"\n{nom}")
    print(f"{'':22} {'sel_avant':>12} {'sel_apres':>12} {'tra_avant':>12} {'tra_apres':>12}")
    print(f"  {'Accuracy':<20} {s1['Acc']:>12} {s2['Acc']:>12} {s3['Acc']:>12} {s4['Acc']:>12}")
    print(f"  {'F1-score':<20} {s1['F1']:>12} {s2['F1']:>12} {s3['F1']:>12} {s4['F1']:>12}")

    return {"sel_avant": s1, "sel_apres": s2, "tra_avant": s3, "tra_apres": s4}

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x.unsqueeze(1))

def evaluer_cnn_h(X_sel, y_sel, X_sel_aug, y_sel_aug,
                  X_tra, y_tra, X_tra_aug, y_tra_aug, epochs=30):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def run(X, y):
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        all_preds, all_true = [], []
        for tr, val in skf.split(X, y):
            X_tr  = torch.FloatTensor(X[tr]).to(device)
            y_tr  = torch.LongTensor(y[tr]).to(device)
            X_val = torch.FloatTensor(X[val]).to(device)
            model = CNN1D(X.shape[1], len(np.unique(y))).to(device)
            opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
            loss  = nn.CrossEntropyLoss()
            loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
            model.train()
            for _ in range(epochs):
                for xb, yb in loader:
                    opt.zero_grad(); loss(model(xb), yb).backward(); opt.step()
            model.eval()
            with torch.no_grad():
                preds = model(X_val).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds); all_true.extend(y[val])
        return {
            "Acc": round(accuracy_score(all_true, all_preds), 4),
            "F1" : round(f1_score(all_true, all_preds, average='weighted'), 4),
        }

    s1 = run(X_sel,     y_sel)
    s2 = run(X_sel_aug, y_sel_aug)
    s3 = run(X_tra,     y_tra)
    s4 = run(X_tra_aug, y_tra_aug)

    print(f"\nCNN 1D")
    print(f"{'':22} {'sel_avant':>12} {'sel_apres':>12} {'tra_avant':>12} {'tra_apres':>12}")
    print(f"  {'Accuracy':<20} {s1['Acc']:>12} {s2['Acc']:>12} {s3['Acc']:>12} {s4['Acc']:>12}")
    print(f"  {'F1-score':<20} {s1['F1']:>12} {s2['F1']:>12} {s3['F1']:>12} {s4['F1']:>12}")

    return {"sel_avant": s1, "sel_apres": s2, "tra_avant": s3, "tra_apres": s4}

In [ ]:
model = KNeighborsClassifier(n_neighbors=5)
resultats_ctgan["KNN"] = evaluer("KNN", model,
    X_sel, y_sel, X_sel_c, y_sel_c,
    X_tra, y_tra, X_tra_c, y_tra_c)

In [ ]:
model = SVC(kernel='rbf', random_state=42)
resultats_ctgan["SVM"] = evaluer("SVM", model,
    X_sel, y_sel, X_sel_c, y_sel_c,
    X_tra, y_tra, X_tra_c, y_tra_c)

In [ ]:
model = DecisionTreeClassifier(random_state=42)
resultats_ctgan["Decision Tree"] = evaluer("Decision Tree", model,
    X_sel, y_sel, X_sel_c, y_sel_c,
    X_tra, y_tra, X_tra_c, y_tra_c)

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
resultats_ctgan["Logistic Regression"] = evaluer("Logistic Regression", model,
    X_sel, y_sel, X_sel_c, y_sel_c,
    X_tra, y_tra, X_tra_c, y_tra_c)

In [ ]:
model = GaussianNB()
resultats_ctgan["Naive Bayes"] = evaluer("Naive Bayes", model,
    X_sel, y_sel, X_sel_c, y_sel_c,
    X_tra, y_tra, X_tra_c, y_tra_c)

In [ ]:
resultats_ctgan["CNN 1D"] = evaluer_cnn_h(
    X_sel,   y_sel,   X_sel_c, y_sel_c,
    X_tra,   y_tra,   X_tra_c, y_tra_c)

In [ ]:
print(f"\n{'TABLEAU RECAPITULATIF CTGAN — F1-score':^80}")
print(f"{'='*80}")
print(f"{'Modele':<22} {'sel_avant':>10} {'sel_apres':>10} {'delta_sel':>10} {'tra_avant':>10} {'tra_apres':>10} {'delta_tra':>10}")
print(f"{'-'*80}")
for nom, s in resultats_ctgan.items():
    d_sel = round(s["sel_apres"]["F1"] - s["sel_avant"]["F1"], 4)
    d_tra = round(s["tra_apres"]["F1"] - s["tra_avant"]["F1"], 4)
    print(f"  {nom:<20} {s['sel_avant']['F1']:>10} {s['sel_apres']['F1']:>10} {d_sel:>10} {s['tra_avant']['F1']:>10} {s['tra_apres']['F1']:>10} {d_tra:>10}")
print(f"{'='*80}")

df_ctgan_result = pd.DataFrame([
    {"Modele": nom,
     "sel_avant": s["sel_avant"]["F1"], "sel_apres": s["sel_apres"]["F1"],
     "delta_sel": round(s["sel_apres"]["F1"] - s["sel_avant"]["F1"], 4),
     "tra_avant": s["tra_avant"]["F1"], "tra_apres": s["tra_apres"]["F1"],
     "delta_tra": round(s["tra_apres"]["F1"] - s["tra_avant"]["F1"], 4)}
    for nom, s in resultats_ctgan.items()
])
df_ctgan_result.to_csv(f'{save_dir}/resultats_ctgan.csv', index=False)
print("Resultats sauvegardes sur Drive")